# 01. Prepare the data

This notebook turns the two raw files into one clean table that every later notebook reads.

| | File | Content |
|---|---|---|
| Input | `ods032.parquet` | Elia solar output, forecasts and capacity, every 15 minutes |
| Input | `weather_belgium.parquet` | Open-Meteo weather, hourly, 11 areas |
| Output | `model_table.parquet` | One row per area per hour, from January 2021: target, raw features, benchmark. Column names are kept exactly as in the two sources |

**Working area.** The team is working on Antwerp, set by `PROVINCE`. The saved table keeps all eleven areas, so nothing has to be rebuilt to look at another one. Later notebooks select the area with `table[table["region"] == PROVINCE]`. Section 5.3 shows the numbers for Antwerp.

**Rules followed here**
- Nothing is learned from the data in this notebook (no averages, no scaling), so nothing can leak from the test period. Missing values are left missing, and are filled later using training data only.
- Every step that removes rows is logged, and the log is shown at the end.

**Steps**
1. Elia: select, convert, aggregate, build the target
2. Weather: blank out bad values
3. Join
4. Flags: daylight and train/test split
5. Final checks and save

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

ELIA_FILE = Path("ods032.parquet")
WEATHER_FILE = Path("weather_belgium.parquet")
OUT_FILE = Path("model_table.parquet")

SAMPLE_START = "2021-01-01"     # first day of the modelling sample (UTC). The weather file starts here
TEST_START = "2025-01-01"       # rows from this day on are the test set; earlier rows are for training and tuning
PROVINCE = "Antwerp"            # the area the team is working on. The saved table keeps all areas

pd.set_option("display.float_format", "{:,.2f}".format)

log = []                        # one entry per step: what was done and how many rows remain
def record(step, rows, note=""):
    removed = (log[-1]["rows"] - rows) if log and log[-1]["table"] == step.split(":")[0] else 0
    log.append({"table": step.split(":")[0], "step": step.split(":", 1)[1].strip(), "rows": rows,
                "removed": removed, "note": note})
    print(f"{step:<55s} {rows:>10,} rows" + (f"   (removed {removed:,})" if removed else ""))

## 1. Elia

### 1.1 Load and select columns

Only the columns that are safe to use are read. Three groups are left out:

| Left out | Why |
|---|---|
| `loadfactor` | It is the target divided by capacity, so it would give the answer away |
| Most recent forecast (3 columns) | Issued just before delivery, long after the trading decision |
| Latest day-ahead forecast (3 columns) | Updated after the 11:00 decision time |
| Week-ahead forecast (3 columns), `resolutioncode` | Not needed |

Elia's 11:00 day-ahead forecast is kept as the **benchmark**, not as a feature.

In [2]:
keep = ["datetime", "region", "measured", "monitoredcapacity",
        "dayahead11hforecast", "dayahead11hconfidence10", "dayahead11hconfidence90"]
elia = pd.read_parquet(ELIA_FILE, columns=keep)
record("Elia: raw file", len(elia))

Elia: raw file                                           2,978,360 rows


### 1.2 Provinces only, UTC, from January 2021

- **Provinces only:** Belgium, Flanders and Wallonia are exact sums of the provinces. Keeping them would count the same output twice.
- **UTC:** the weather file is in UTC, so both tables must share one clock.
- **January 2021:** the earliest date both sources cover. Elia starts in October 2020 and the weather file in January 2021.

In [3]:
AGGREGATES = ["Belgium", "Flanders", "Wallonia"]
elia = elia[~elia["region"].isin(AGGREGATES)]
record("Elia: drop Belgium, Flanders, Wallonia", len(elia))

elia["datetime"] = pd.to_datetime(elia["datetime"], utc=True)      # same column, now in UTC
elia = elia[elia["datetime"] >= pd.Timestamp(SAMPLE_START, tz="UTC")]
record(f"Elia: keep from {SAMPLE_START}", len(elia))

dups = elia.duplicated(["region", "datetime"]).sum()
elia = elia.drop_duplicates(["region", "datetime"])
record("Elia: drop duplicate (area, time) rows", len(elia), f"{dups} duplicates")
print(f"Areas: {sorted(elia['region'].unique())}")

Elia: drop Belgium, Flanders, Wallonia                   2,340,140 rows   (removed 638,220)


Elia: keep from 2021-01-01                               2,242,900 rows   (removed 97,240)


Elia: drop duplicate (area, time) rows                   2,242,900 rows
Areas: ['Antwerp', 'Brussels', 'East-Flanders', 'Flemish-Brabant', 'Hainaut', 'Limburg', 'Liège', 'Luxembourg', 'Namur', 'Walloon-Brabant', 'West-Flanders']


### 1.3 From 15 minutes to hourly

Weather is hourly, so the four quarter-hour values of each hour are averaged.

Two details matter:
- **Labelling.** Open-Meteo's radiation at hour *T* is the average over the hour ending at *T*. Each Elia hour is therefore labelled with its **end** time, so both tables describe the same 60 minutes.
- **Completeness.** An hour is kept if all four quarter-hour values of measured output are present. Averaging three out of four would bias a daytime hour. One exception: if a value is missing but every other value in that hour is zero, the hour is night and is kept as zero. The cell below reports how many missing values there are, when they occur, and how many of them are night hours of this kind.

In [4]:
missing_15 = elia["measured"].isna()
print(f"Quarter-hour rows with missing measured output: {missing_15.sum():,} ({missing_15.mean():.2%})")
local = elia.loc[missing_15, "datetime"].dt.tz_convert("Europe/Brussels")
print(f"  local times of the missing values: {local.dt.strftime('%H:%M').value_counts().to_dict()}")
print(f"  from {local.min():%d %b %Y} to {local.max():%d %b %Y}")

value_cols = ["measured", "monitoredcapacity", "dayahead11hforecast",
              "dayahead11hconfidence10", "dayahead11hconfidence90"]
grouped = (elia.set_index("datetime").groupby("region")[value_cols]
           .resample("1h", label="right", closed="left"))
hourly = grouped.mean()
n_valid = grouped["measured"].count()["measured"]      # how many of the four values exist
max_present = grouped["measured"].max()["measured"]    # largest of the values that exist
record("Elia: aggregate to hourly", len(hourly))

complete = n_valid == 4
night_gap = (n_valid < 4) & (n_valid > 0) & (max_present == 0)
print(f"Hours with a missing value: {(~complete).sum():,}, of which night hours kept as zero: {night_gap.sum():,}")
hourly = hourly[complete | night_gap].reset_index()
record("Elia: drop incomplete daytime hours", len(hourly), f"{night_gap.sum()} night hours with one missing value kept as zero")

Quarter-hour rows with missing measured output: 7,018 (0.31%)
  local times of the missing values: {'23:45': 7018}
  from 16 Nov 2022 to 15 Oct 2024


Elia: aggregate to hourly                                  560,725 rows   (removed 1,682,175)
Hours with a missing value: 7,018, of which night hours kept as zero: 7,018
Elia: drop incomplete daytime hours                        560,725 rows


### 1.4 Build the target

The target is the **load factor**: measured output divided by monitored capacity, a number between 0 and 1.

Why not predict MW directly? Installed capacity more than doubles over the sample. In MW, the same sunny afternoon produces far more in 2026 than in 2021, for a reason unrelated to weather. Dividing by capacity removes that trend, so all years are comparable. A forecast in MW is recovered by multiplying back by capacity.

All Elia columns keep the names they have in the raw file. The load factor is the only column added here, under the name `lf`, to avoid confusion with Elia's own `loadfactor` column (which is in percent and at 15 minute resolution).

In [5]:
hourly["lf"] = hourly["measured"] / hourly["monitoredcapacity"]

impossible = (hourly["lf"] < 0) | (hourly["lf"] > 1) | hourly["monitoredcapacity"].le(0)
print(f"Hours with a load factor below 0 or above 1: {impossible.sum()}")
if impossible.any():
    bad = hourly[impossible]
    display(bad.groupby(["region", bad["datetime"].dt.date]).agg(
        hours=("lf", "size"), highest_lf_pct=("lf", lambda s: s.max() * 100),
        monitoredcapacity=("monitoredcapacity", "first"), highest_measured=("measured", "max")))
hourly = hourly[~impossible]
record("Elia: drop impossible load factors", len(hourly))

print(f"Highest load factor: {hourly['lf'].max():.1%}   Benchmark missing in {hourly['dayahead11hforecast'].isna().mean():.2%} of hours")

Hours with a load factor below 0 or above 1: 23


hours  highest_lf_pct  monitoredcapacity  \
region   datetime                                               
Brussels 2022-03-21      5          146.55              54.59   
         2022-03-22      6          147.63              54.59   
         2022-03-23      6          148.44              54.59   
         2022-03-24      6          150.11              54.59   

                     highest_measured  
region   datetime                      
Brussels 2022-03-21             80.00  
         2022-03-22             80.58  
         2022-03-23             81.03  
         2022-03-24             81.94

Elia: drop impossible load factors                         560,702 rows   (removed 23)
Highest load factor: 96.1%   Benchmark missing in 0.33% of hours


## 2. Weather

### 2.1 Load and restrict to the sample period

In [6]:
wx = pd.read_parquet(WEATHER_FILE).drop(columns=["latitude", "longitude"])
record("Weather: raw file", len(wx))
wx = wx[wx["time"] >= pd.Timestamp(SAMPLE_START, tz="UTC")]
record(f"Weather: keep from {SAMPLE_START}", len(wx))

WEATHER_FEATURES = [c for c in wx.columns if c not in ("time", "region")]
print(f"{len(WEATHER_FEATURES)} weather features: {WEATHER_FEATURES}")

Weather: raw file                                          560,736 rows
Weather: keep from 2021-01-01                              560,736 rows
16 weather features: ['shortwave_radiation', 'direct_radiation', 'diffuse_radiation', 'direct_normal_irradiance', 'cloud_cover', 'cloud_cover_low', 'cloud_cover_mid', 'cloud_cover_high', 'temperature_2m', 'dew_point_2m', 'relative_humidity_2m', 'surface_pressure', 'precipitation', 'wind_speed_10m', 'wind_gusts_10m', 'wind_direction_10m']


### 2.2 Blank out implausible values

Each variable has a range that is physically plausible for Belgium. A value outside it is set to missing. The row is kept, since its other variables are still good.

The known case is 26 to 28 June 2024, when relative humidity and dew point are corrupted in every area.

In [7]:
LIMITS = {   # column: (lowest plausible, highest plausible)
    "shortwave_radiation": (0, 1100), "direct_radiation": (0, 1000), "diffuse_radiation": (0, 700), "direct_normal_irradiance": (0, 1100),
    "cloud_cover": (0, 100), "cloud_cover_low": (0, 100), "cloud_cover_mid": (0, 100), "cloud_cover_high": (0, 100),
    "temperature_2m": (-25, 45), "dew_point_2m": (-25, 30), "relative_humidity_2m": (10, 100), "surface_pressure": (900, 1060),
    "precipitation": (0, 60), "wind_speed_10m": (0, 130), "wind_gusts_10m": (0, 200), "wind_direction_10m": (0, 360),
}
blanked = {}
for c in WEATHER_FEATURES:
    lo, hi = LIMITS[c]
    bad = (wx[c] < lo) | (wx[c] > hi)
    if bad.any():
        blanked[c] = {"values blanked": int(bad.sum()),
                      "first": wx.loc[bad, "time"].min().strftime("%Y-%m-%d"),
                      "last": wx.loc[bad, "time"].max().strftime("%Y-%m-%d")}
        wx.loc[bad, c] = np.nan
record("Weather: blank implausible values (rows kept)", len(wx), f"{sum(v['values blanked'] for v in blanked.values())} values")
pd.DataFrame(blanked).T if blanked else "No implausible values"

Weather: blank implausible values (rows kept)              560,736 rows


,values blanked,first,last
dew_point_2m,660,2024-06-26,2024-06-28
relative_humidity_2m,660,2024-06-26,2024-06-28


### 2.3 A known weakness that is kept, not fixed

From mid 2024, wind speed, wind gusts and mid-level cloud cover sit at a lower level, which points to a change in the weather model behind the archive (see `explore_weather.ipynb`, section 4.4). The training years and the test years therefore do not measure these three variables in quite the same way.

They are kept, because they are raw features of the registered dataset. They are listed here so that the modelling notebook can test whether results depend on them.

In [8]:
SHIFTED_FEATURES = ["wind_speed_10m", "wind_gusts_10m", "cloud_cover_mid"]
before = wx[wx["time"] < "2024-07-01"][SHIFTED_FEATURES].mean()
after = wx[wx["time"] >= "2024-07-01"][SHIFTED_FEATURES].mean()
pd.DataFrame({"mean before July 2024": before, "mean from July 2024": after, "change %": (after / before - 1) * 100})

,mean before July 2024,mean from July 2024,change %
wind_speed_10m,14.54,10.61,-27.03
wind_gusts_10m,27.96,20.77,-25.74
cloud_cover_mid,32.84,25.11,-23.55


## 3. Join

An inner join on area and hour: a row survives only if both the target and the weather exist for it.

In [9]:
table = (hourly.merge(wx, left_on=["region", "datetime"], right_on=["region", "time"], how="inner", validate="one_to_one")
         .drop(columns="time"))
record("Joined: Elia hours matched with weather", len(table))
print(f"Elia hours without weather: {len(hourly) - len(table):,}")

Joined: Elia hours matched with weather                    560,702 rows
Elia hours without weather: 0


## 4. Flags

### 4.1 Daylight

At night output is zero by definition, so there is nothing to forecast. Night rows are **kept** in the table, since later notebooks need them to build lagged features, but they are flagged so they can be left out of training and scoring.

The flag uses the weather forecast (irradiance above zero), which is known in advance. It must not use measured output, which is not.

In [10]:
table["is_daylight"] = table["shortwave_radiation"] > 0

night = table[~table["is_daylight"]]
print(f"Daylight hours: {table['is_daylight'].sum():,} ({table['is_daylight'].mean():.1%} of rows)")
print(f"Share of all measured energy that falls in hours flagged as night: "
      f"{night['measured'].sum() / table['measured'].sum():.4%}")
print(f"Highest load factor in an hour flagged as night: {night['lf'].max():.2%}")

Daylight hours: 300,135 (53.5% of rows)
Share of all measured energy that falls in hours flagged as night: 0.0017%
Highest load factor in an hour flagged as night: 1.53%


### 4.2 Train and test

The split is by time, never random. A random split would let the model see days on both sides of each test day, which is information from the future.

- **train:** before 1 January 2025. All model fitting and tuning happens here (with time-ordered cross-validation inside it).
- **test:** from 1 January 2025. Looked at once, at the end.

In [11]:
table["split"] = np.where(table["datetime"] < pd.Timestamp(TEST_START, tz="UTC"), "train", "test")
table.groupby("split").agg(rows=("lf", "size"), daylight_rows=("is_daylight", "sum"),
                           first=("datetime", "min"), last=("datetime", "max")).loc[["train", "test"]]

,rows,daylight_rows,first,last
split,,,,
train,385670,204848,2021-01-01 01:00:00+00:00,2024-12-31 23:00:00+00:00
test,175032,95287,2025-01-01 00:00:00+00:00,2026-10-25 23:00:00+00:00


## 5. Final checks and save

### 5.1 Column roles

In [12]:
RAW_FEATURES = WEATHER_FEATURES + ["region", "monitoredcapacity"]
roles = {"datetime": "key (UTC, end of the hour)", "region": "key and raw feature",
         "lf": "TARGET, added here (measured / monitoredcapacity, 0 to 1)",
         "measured": "target in MW (Elia name)", "monitoredcapacity": "raw feature (Elia name)",
         "dayahead11hforecast": "benchmark in MW (Elia name)", "dayahead11hconfidence10": "benchmark in MW (Elia name)",
         "dayahead11hconfidence90": "benchmark in MW (Elia name)",
         "is_daylight": "flag, added here", "split": "flag, added here"}
roles.update({c: "raw feature (Open-Meteo name)" for c in WEATHER_FEATURES})

first = ["datetime", "region", "split", "is_daylight", "lf", "measured", "monitoredcapacity",
         "dayahead11hforecast", "dayahead11hconfidence10", "dayahead11hconfidence90"]
table = table[first + WEATHER_FEATURES].sort_values(["region", "datetime"]).reset_index(drop=True)

print(f"Raw features before any feature engineering: {len(RAW_FEATURES)} (the project requires at least 15)")
overview = pd.DataFrame({"role": [roles[c] for c in table.columns],
                         "missing %": [table[c].isna().mean() * 100 for c in table.columns]}, index=table.columns)
overview

Raw features before any feature engineering: 18 (the project requires at least 15)


,role,missing %
datetime,"key (UTC, end of the hour)",0.00
region,key and raw feature,0.00
split,"flag, added here",0.00
is_daylight,"flag, added here",0.00
lf,"TARGET, added here (measured / monitoredcapaci...",0.00
measured,target in MW (Elia name),0.00
monitoredcapacity,raw feature (Elia name),0.00
dayahead11hforecast,benchmark in MW (Elia name),0.33
dayahead11hconfidence10,benchmark in MW (Elia name),0.33
dayahead11hconfidence90,benchmark in MW (Elia name),0.33


### 5.2 Rows per area and coverage

In [13]:
expected = int((table["datetime"].max() - table["datetime"].min()) / pd.Timedelta("1h")) + 1
cov = table.groupby("region").agg(rows=("lf", "size"), first=("datetime", "min"), last=("datetime", "max"),
                                  mean_load_factor_pct=("lf", lambda s: s.mean() * 100))
cov["coverage %"] = cov["rows"] / expected * 100
cov

,rows,first,last,mean_load_factor_pct,coverage %
region,,,,,
Antwerp,50975,2021-01-01 01:00:00+00:00,2026-10-25 23:00:00+00:00,11.05,100.00
Brussels,50952,2021-01-01 01:00:00+00:00,2026-10-25 23:00:00+00:00,11.05,99.95
East-Flanders,50975,2021-01-01 01:00:00+00:00,2026-10-25 23:00:00+00:00,11.23,100.00
Flemish-Brabant,50975,2021-01-01 01:00:00+00:00,2026-10-25 23:00:00+00:00,11.11,100.00
Hainaut,50975,2021-01-01 01:00:00+00:00,2026-10-25 23:00:00+00:00,11.05,100.00
Limburg,50975,2021-01-01 01:00:00+00:00,2026-10-25 23:00:00+00:00,11.03,100.00
Liège,50975,2021-01-01 01:00:00+00:00,2026-10-25 23:00:00+00:00,10.84,100.00
Luxembourg,50975,2021-01-01 01:00:00+00:00,2026-10-25 23:00:00+00:00,10.41,100.00
Namur,50975,2021-01-01 01:00:00+00:00,2026-10-25 23:00:00+00:00,10.83,100.00


### 5.3 The working area

The rows later notebooks will use when they select `PROVINCE`.

In [14]:
mine = table[table["region"] == PROVINCE]
print(f"{PROVINCE}: {len(mine):,} hourly rows, {mine['is_daylight'].sum():,} in daylight")
print(f"Period: {mine['datetime'].min()}  to  {mine['datetime'].max()}")
print(f"Monitored capacity: {mine['monitoredcapacity'].iloc[0]:,.0f} MW at the start, {mine['monitoredcapacity'].iloc[-1]:,.0f} MW at the end")
print(f"Mean load factor in daylight: {mine.loc[mine['is_daylight'], 'lf'].mean():.1%}   Highest: {mine['lf'].max():.1%}")
mine.groupby("split").agg(rows=("lf", "size"), daylight_rows=("is_daylight", "sum"),
                          first=("datetime", "min"), last=("datetime", "max")).loc[["train", "test"]]

Antwerp: 50,975 hourly rows, 27,293 in daylight
Period: 2021-01-01 01:00:00+00:00  to  2026-10-25 23:00:00+00:00
Monitored capacity: 777 MW at the start, 2,029 MW at the end
Mean load factor in daylight: 20.6%   Highest: 91.8%


,rows,daylight_rows,first,last
split,,,,
train,35063,18611,2021-01-01 01:00:00+00:00,2024-12-31 23:00:00+00:00
test,15912,8682,2025-01-01 00:00:00+00:00,2026-10-25 23:00:00+00:00


### 5.4 What each step removed

This table goes into the data section of the report. The counts are for all areas.

In [15]:
steps = pd.DataFrame(log)[["table", "step", "rows", "removed", "note"]]
steps

,table,step,rows,removed,note
0,Elia,raw file,2978360,0,
1,Elia,"drop Belgium, Flanders, Wallonia",2340140,638220,
2,Elia,keep from 2021-01-01,2242900,97240,
3,Elia,"drop duplicate (area, time) rows",2242900,0,0 duplicates
4,Elia,aggregate to hourly,560725,1682175,
5,Elia,drop incomplete daytime hours,560725,0,7018 night hours with one missing value kept a...
6,Elia,drop impossible load factors,560702,23,
7,Weather,raw file,560736,0,
8,Weather,keep from 2021-01-01,560736,0,
9,Weather,blank implausible values (rows kept),560736,0,1320 values


In [16]:
assert not table.duplicated(["region", "datetime"]).any(), "duplicate rows"
assert table["lf"].between(0, 1).all(), "target outside 0 to 1"
assert table["lf"].notna().all(), "missing target"
assert (table["datetime"] >= pd.Timestamp(SAMPLE_START, tz="UTC")).all()

table.to_parquet(OUT_FILE, index=False)
print(f"Saved {OUT_FILE}: {len(table):,} rows, {table.shape[1]} columns, {OUT_FILE.stat().st_size / 1e6:.1f} MB")
print(f"Period: {table['datetime'].min()}  to  {table['datetime'].max()}")

Saved model_table.parquet: 560,702 rows, 26 columns, 20.2 MB
Period: 2021-01-01 01:00:00+00:00  to  2026-10-25 23:00:00+00:00


## What the next notebook receives

`model_table.parquet`, one row per area per hour:

- **Column names are the source names.** Elia columns are named as in `ods032.parquet`, and weather columns as in the Open-Meteo API. Only three columns are added: `lf`, `is_daylight` and `split`.
- **`lf`** is the target: `measured` divided by `monitoredcapacity`.
- **`dayahead11hforecast`, `dayahead11hconfidence10`, `dayahead11hconfidence90`** are Elia's benchmark, in MW. Divide by `monitoredcapacity` to compare with `lf`. They must never be used as model inputs.
- **`region`** selects the area. The team is working on Antwerp: `table[table["region"] == "Antwerp"]`.
- **`split`** and **`is_daylight`** say which rows to train on and which to score.
- **Weather columns, `region` and `monitoredcapacity`** are the raw features.
- **Missing values** in weather columns are still missing. Fill them using statistics computed on `split == "train"` only, or use a model that handles missing values natively.